# CodeBERT: exploratory raw-label classification
Raw label meanings and repository/near-duplicate leakage remain unverified. These results are not vulnerability or patch-reliability claims.


In [ ]:
from pathlib import Path
import sys, subprocess, importlib.util, json
input_root = Path('/kaggle/input')
candidates = sorted(p.parent for p in input_root.rglob('train.csv') if (p.parent / 'validation.csv').is_file() and (p.parent / 'train_codebert.py').is_file())
if len(candidates) != 1:
    raise RuntimeError(f'Attach one private dataset with train.csv, validation.csv and train_codebert.py together; found {len(candidates)}.')
data_dir = candidates[0]
print('Dataset:', data_dir)
print('Python:', sys.version)


In [ ]:
if not Path('/kaggle/working').is_dir():
    raise RuntimeError('This notebook must run in Kaggle.')
try:
    subprocess.run(['nvidia-smi'], check=True)
except (FileNotFoundError, subprocess.CalledProcessError) as exc:
    raise RuntimeError('Enable a Kaggle GPU accelerator before continuing.') from exc
missing = []
if importlib.util.find_spec('torch') is None: missing.append('torch>=2.2,<3')
if importlib.util.find_spec('transformers') is None: missing.append('transformers>=4.40,<5')
if importlib.util.find_spec('sklearn') is None: missing.append('scikit-learn>=1.3,<2')
if missing:
    subprocess.run([sys.executable, '-m', 'pip', 'install', *missing], check=True)
import torch, transformers, sklearn
print('PyTorch:', torch.__version__, '| Transformers:', transformers.__version__, '| scikit-learn:', sklearn.__version__)
if not torch.cuda.is_available():
    raise RuntimeError('CUDA unavailable in PyTorch. Select a Kaggle GPU environment with CUDA-compatible PyTorch.')
print('GPU:', torch.cuda.get_device_name(0))


In [ ]:
output = Path('/kaggle/working/codebert-experiment')
if output.exists():
    raise FileExistsError(f'{output} already exists. Download or move the existing output before rerunning.')
command = [sys.executable, str(data_dir / 'train_codebert.py'), '--experimental-raw-labels', '--data-dir', str(data_dir), '--output-dir', str(output)]
# The script downloads pretrained CodeBERT weights from Hugging Face on Kaggle.
subprocess.run(command, check=True)


In [ ]:
with (output / 'metrics.json').open(encoding='utf-8') as stream:
    metrics = json.load(stream)
print('EXPERIMENTAL raw-label validation metrics:')
print(json.dumps(metrics, indent=2))
print('Model, tokenizer and metrics:', output)
